<a href="https://colab.research.google.com/github/myoohit/IIT-Madras-assignment/blob/main/DeepLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

root = '/content/drive/MyDrive/aksharantar_sampled'
lang = 'hin'
paths = {
    'train': os.path.join(root, lang, f'{lang}_train.csv'),
    'valid': os.path.join(root, lang, f'{lang}_valid.csv'),
    'test':  os.path.join(root, lang, f'{lang}_test.csv')
}

for p in paths.values():
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing dataset file: {p}")

train_df = pd.read_csv(paths['train'])
valid_df = pd.read_csv(paths['valid'])
test_df  = pd.read_csv(paths['test'])

train_data = list(zip(train_df.iloc[:,0].astype(str), train_df.iloc[:,1].astype(str)))
valid_data = list(zip(valid_df.iloc[:,0].astype(str), valid_df.iloc[:,1].astype(str)))
test_data  = list(zip(test_df.iloc[:,0].astype(str),  test_df.iloc[:,1].astype(str)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

embed_dim = 128
hidden_dim = 256
num_enc_layers = 1
num_dec_layers = 1
cell_type = 'GRU'
batch_size = 64
lr = 1e-3
num_epochs = 20
tf_ratio = 0.5
save_path = f'/content/drive/MyDrive/akshantar_models/{lang}_seq2seq_new.pth'
os.makedirs(os.path.dirname(save_path), exist_ok=True)

class CharVocab:
    def __init__(self, charset, name):
        self.name = name
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2}
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>"}
        self.n = 3
        for ch in sorted(charset):
            if ch not in self.stoi:
                self.stoi[ch] = self.n
                self.itos[self.n] = ch
                self.n += 1

def make_vocabs(pairs):
    src_set = set(ch for s, _ in pairs for ch in s)
    tgt_set = set(ch for _, t in pairs for ch in t)
    return CharVocab(src_set, 'latin'), CharVocab(tgt_set, 'native')

src_vocab, tgt_vocab = make_vocabs(train_data)
max_src_len = max(len(s) for s, _ in train_data + valid_data + test_data) + 1
max_tgt_len = max(len(t) for _, t in train_data + valid_data + test_data) + 2

class TranslitDataset(Dataset):
    def __init__(self, pairs, src_v, tgt_v, max_src, max_tgt):
        self.pairs = pairs
        self.src_v = src_v
        self.tgt_v = tgt_v
        self.max_src = max_src
        self.max_tgt = max_tgt

    def __len__(self):
        return len(self.pairs)

    def encode_src(self, s):
        ids = [self.src_v.stoi.get(c, 0) for c in s] + [self.src_v.stoi["<EOS>"]]
        ids += [0] * (self.max_src - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def encode_tgt(self, t):
        ids = [self.tgt_v.stoi["<SOS>"]] + [self.tgt_v.stoi.get(c, 0) for c in t] + [self.tgt_v.stoi["<EOS>"]]
        ids += [0] * (self.max_tgt - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __getitem__(self, idx):
        s, t = self.pairs[idx]
        return self.encode_src(s).to(device), self.encode_tgt(t).to(device)

train_ds = TranslitDataset(train_data, src_vocab, tgt_vocab, max_src_len, max_tgt_len)
valid_ds = TranslitDataset(valid_data, src_vocab, tgt_vocab, max_src_len, max_tgt_len)
test_ds  = TranslitDataset(test_data,  src_vocab, tgt_vocab, max_src_len, max_tgt_len)

train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=batch_size)
test_dl  = DataLoader(test_ds,  batch_size=batch_size)

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, layers, mode):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if mode == 'GRU':
            self.rnn = nn.GRU(emb_dim, hid_dim, layers, batch_first=True)
        elif mode == 'LSTM':
            self.rnn = nn.LSTM(emb_dim, hid_dim, layers, batch_first=True)
        else:
            self.rnn = nn.RNN(emb_dim, hid_dim, layers, batch_first=True)

    def forward(self, x):
        x = self.emb(x)
        _, hidden = self.rnn(x)
        return hidden

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, layers, mode):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if mode == 'GRU':
            self.rnn = nn.GRU(emb_dim, hid_dim, layers, batch_first=True)
        elif mode == 'LSTM':
            self.rnn = nn.LSTM(emb_dim, hid_dim, layers, batch_first=True)
        else:
            self.rnn = nn.RNN(emb_dim, hid_dim, layers, batch_first=True)
        self.fc = nn.Linear(hid_dim, vocab_size)

    def forward(self, x, hidden):
        emb = self.emb(x)
        out, hidden = self.rnn(emb, hidden)
        return self.fc(out[:, 0, :]), hidden

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec, out_dim):
        super().__init__()
        self.enc = enc
        self.dec = dec
        self.out_dim = out_dim

    def forward(self, src, tgt, tf_ratio=0.5):
        bsz, seq_len = tgt.shape
        preds = torch.zeros(bsz, seq_len, self.out_dim, device=device)
        hidden = self.enc(src)
        dec_input = tgt[:, 0].unsqueeze(1)

        for t in range(1, seq_len):
            output, hidden = self.dec(dec_input, hidden)
            preds[:, t, :] = output
            teacher = random.random() < tf_ratio
            top = output.argmax(1)
            dec_input = tgt[:, t].unsqueeze(1) if teacher else top.unsqueeze(1)
        return preds

def init_weights(m):
    if hasattr(m, "weight") and m.weight.dim() > 1:
        nn.init.xavier_uniform_(m.weight.data)

enc = Encoder(src_vocab.n, embed_dim, hidden_dim, num_enc_layers, cell_type).to(device)
dec = Decoder(tgt_vocab.n, embed_dim, hidden_dim, num_dec_layers, cell_type).to(device)
model = Seq2Seq(enc, dec, tgt_vocab.n).to(device)
model.apply(init_weights)

optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.stoi["<PAD>"])

def train_one_epoch(model, loader, optimizer, loss_fn, clip=1):
    model.train()
    epoch_loss = 0
    for src, tgt in loader:
        optimizer.zero_grad()
        out = model(src, tgt, tf_ratio)
        out_dim = out.shape[-1]
        loss = loss_fn(out[:, 1:, :].reshape(-1, out_dim), tgt[:, 1:].reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def evaluate(model, loader, loss_fn):
    model.eval()
    total = 0
    with torch.no_grad():
        for src, tgt in loader:
            out = model(src, tgt, 0)
            out_dim = out.shape[-1]
            loss = loss_fn(out[:, 1:, :].reshape(-1, out_dim), tgt[:, 1:].reshape(-1))
            total += loss.item()
    return total / len(loader)

for ep in range(1, num_epochs + 1):
    tr_loss = train_one_epoch(model, train_dl, optimizer, criterion)
    val_loss = evaluate(model, valid_dl, criterion)
    print(f"Epoch {ep:02d} | Train: {tr_loss:.4f} | Val: {val_loss:.4f}")
    torch.save({
        'epoch': ep,
        'model': model.state_dict(),
        'opt': optimizer.state_dict()
    }, save_path)

def predict(model, word, src_v, tgt_v, max_len):
    model.eval()
    with torch.no_grad():
        seq = [src_v.stoi.get(c, 0) for c in word] + [src_v.stoi["<EOS>"]]
        seq += [0] * (max_src_len - len(seq))
        src_tensor = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
        hidden = model.enc(src_tensor)
        dec_input = torch.tensor([[tgt_v.stoi["<SOS>"]]], dtype=torch.long, device=device)
        result = []
        for _ in range(max_len):
            out, hidden = model.dec(dec_input, hidden)
            top = out.argmax(1).item()
            ch = tgt_v.itos[top]
            if ch in ("<EOS>", "<PAD>"):
                break
            result.append(ch)
            dec_input = torch.tensor([[top]], dtype=torch.long, device=device)
        return ''.join(result)

print("\nSample Predictions:")
for i in range(10):
    s, t = test_data[i]
    out = predict(model, s, src_vocab, tgt_vocab, max_tgt_len)
    print(f"{s:<12} | {t:<12} | {out:<12}")

def token_accuracy(model, loader, vocab):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for src, tgt in loader:
            out = model(src, tgt, 0)
            preds = out.argmax(-1)
            tgt_trim = tgt[:, 1:]
            pred_trim = preds[:, 1:]
            mask = tgt_trim != vocab.stoi["<PAD>"]
            correct += ((pred_trim == tgt_trim) & mask).sum().item()
            total += mask.sum().item()
    return correct / total if total else 0

acc = token_accuracy(model, test_dl, tgt_vocab)
print(f"\nToken-level Test Accuracy: {acc * 100:.2f}%")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Epoch 01 | Train: 2.7437 | Val: 2.3883
Epoch 02 | Train: 1.7627 | Val: 1.6119
Epoch 03 | Train: 1.2574 | Val: 1.3828
Epoch 04 | Train: 1.0509 | Val: 1.2646
Epoch 05 | Train: 0.9404 | Val: 1.2273
Epoch 06 | Train: 0.8557 | Val: 1.1973
Epoch 07 | Train: 0.8081 | Val: 1.1763
Epoch 08 | Train: 0.7494 | Val: 1.1741
Epoch 09 | Train: 0.7160 | Val: 1.1436
Epoch 10 | Train: 0.6695 | Val: 1.1531
Epoch 11 | Train: 0.6528 | Val: 1.1712
Epoch 12 | Train: 0.6188 | Val: 1.1350
Epoch 13 | Train: 0.5851 | Val: 1.1533
Epoch 14 | Train: 0.5629 | Val: 1.1582
Epoch 15 | Train: 0.5495 | Val: 1.1694
Epoch 16 | Train: 0.5189 | Val: 1.1674
Epoch 17 | Train: 0.5048 | Val: 1.1774
Epoch 18 | Train: 0.4857 | Val: 1.1691
Epoch 19 | Train: 0.4671 | Val: 1.2052
Epoch 20 | Train: 0.4530 | Val: 1.2147

Sample Predictions:
sikhaaega    | सिखाएगा      | सिखाएगा     
learn        | लर्न        